# 00 Â· Observe the workbench

Check the arm, both cameras, GPU and available kernels before running a model.
This notebook reads cached workbench state and images. It never claims control,
opens hardware devices, changes torque, or starts a model call.

**Kernel:** `Ramion Â· setup / Codex`  
**Task:** move the stuffed mouse onto the left gray area, right gray area, or between them.

If imports fail, run `bash notebooks/setup.sh core` in the repository and select the matching kernel.


In [ ]:
import os
from pathlib import Path
from soarm_lab.notebooks.client import WorkbenchClient

REPO = next((p for base in (Path.cwd(), *Path.cwd().parents)
             for p in (base, base / 'apps' / 'arm-workbench')
             if (p / 'packages' / 'soarm').is_dir()), None)
if REPO is None:
    raise RuntimeError('Open this notebook from the Ramion checkout.')
WORKSPACE = REPO / 'workspace' / 'notebooks'
WORKSPACE.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('HF_HOME', str(REPO / 'workspace' / 'huggingface'))
client = WorkbenchClient(os.environ.get('RAMION_WORKBENCH_URL', 'http://100.85.56.14:8765'))
TASK = 'Move the stuffed toy mouse onto the left gray area.'


## 1. Check the connection

The workbench remains the sole owner of the arm and camera devices. A browser owner
must release control before a later notebook motion session can begin.


In [ ]:
report = client.preflight()
report


## 2. Inspect both live views

Identify the stuffed mouse and both gray areas in **overhead**. Check that **wrist**
shows the gripper. The views below are snapshots, not a continuously updating video.
Rerun this cell to refresh them.


In [ ]:
snapshot = client.snapshot()
client.show(snapshot)
print('Joint feedback:', snapshot.joints)
print('Camera ages at acquisition:', snapshot.ages)


## 3. Check local tools without opening credentials

The server has an RTX 3060 with 12 GB VRAM. This check reports executable availability only. Each operator supplies a dedicated
Codex login profile or enters their own Dream key in the corresponding notebook.


In [ ]:
import shutil
import subprocess
import sys

print('Python:', sys.executable)
print('Codex executable:', shutil.which('codex'))
print('Cloud credentials: bring your own; configure them in notebook 02 or 03.')
if shutil.which('nvidia-smi'):
    completed = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
        check=True, capture_output=True, text=True,
    )
    print(completed.stdout.strip())
else:
    print('nvidia-smi is unavailable; resolve GPU access before the local VLA notebook.')


## 4. Check whether this deployment can run model sessions

The current development source adds coordinated actions and bounded model sessions.
The separately deployed workbench must contain that update before arming a notebook.
A missing capability does not prevent camera viewing or inference previews.

Deployment and physical motion tests belong to a **supervised handoff**. Do not launch
another hardware process or restart the running arm service from these notebooks.


In [ ]:
capabilities = client.state().get('capabilities', {})
print('Capabilities:', capabilities)
if not capabilities.get('model_control'):
    print('Inference previews are available. Model motion needs the updated workbench deployment.')
else:
    print('Model control API is present. Motion still requires explicit arming in a model notebook.')


## Choose the next notebook

| Notebook | Use it for | Kernel |
| --- | --- | --- |
| `01_local_vla.ipynb` | Local SmolVLA inference and latency | Ramion Â· SmolVLA GPU |
| `02_codex.ipynb` | Taught skills selected by your explicitly selected Codex profile | Ramion Â· setup / Codex |
| `03_dream.ipynb` | Dream-1 action preview and bounded episodes | Ramion Â· Dream-1 |

Keep the workbench's STOP available during any later physical session. STOP is a
position hold with torque retained, not a power disconnect.
